In [11]:
#cross-encoder/ms-marco-MiniLM-L-12-v2
import regex as re
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy  as np

from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator, CECorrelationEvaluator

In [2]:

# from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator

# 1. Load the small base model
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2', num_labels=1)


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [6]:

# Load your data
df = pd.read_csv("final_dataframe.csv")

def clean_text(x):
    x = re.sub(r'\s+', ' ', x)
    x = re.sub(r'http\S+', '', x)
    x = re.sub(r'[^A-Za-z0-9 .,]', '', x)
    return x.strip()

df["resume"] = df["resume"].apply(clean_text)
df["jd"] = df["jd"].apply(clean_text)
df["new_ats"] = df["new_ats"]/100.0
# Assuming your dataframe has columns: 'resume', 'job_description', 'ats_score'
X = df[['resume', 'jd']]
y = df['new_ats']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [18]:
# 2. Prepare Data into InputExamples
train_samples = []
for index, row in X_train.iterrows():
    score = y_train.loc[index]
    # Ensure the score is a float
    train_samples.append(InputExample(texts=[row['resume'], row['jd']], label=float(score)))

# The DataLoader wraps our training samples
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)


# 3. Prepare test/eavaluation data
test_samples = []
for index, row in X_test.iterrows():
    score = y_test.loc[index]
    test_samples.append(InputExample(texts=[row['resume'], row['jd']], label=float(score)))


# 4. Fine-Tune the model
evaluator = CECorrelationEvaluator.from_input_examples(test_samples, name='test')

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=3,
    evaluation_steps=1000,
    warmup_steps=100,
    output_path="Models/ATS-Finetuned"
)
model.save("Models/ATS-Finetuned-final")


D:\DS Projects\Testing\ats_env\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [12]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.0


ERROR! Session/line number was not unique in database. History logging moved to new session 356
Looking in indexes: https://download.pytorch.org/whl/rocm6.0
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)

[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torchaudio


None
